# M1-03 — CNN 3 blocks

Code tương ứng với `m1_03.py`. Chạy các cell từ trên xuống: import, cấu hình, xử lý ảnh, load dữ liệu, tạo model, train/validation và test.

Train tối đa 50 epoch, dừng khi validation accuracy không tăng trong 15 epoch liên tiếp. Test dùng trọng số có validation accuracy cao nhất giữ trong RAM. Notebook chưa được chạy.


In [1]:
import os

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from sklearn.metrics import classification_report

from metrics import calculate_metrics

In [2]:
# 1. Cấu hình
DATA_DIR = "../../data/VN_trash_classification_preprocessing"
CHECKPOINT_DIR = "checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [3]:
# 2. Xử lý ảnh
normalize = transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    normalize,
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    normalize,
])


In [4]:
# 3. Load dữ liệu
train_dataset = ImageFolder(DATA_DIR + "/train", transform=train_transform)
val_dataset = ImageFolder(DATA_DIR + "/val", transform=test_transform)
test_dataset = ImageFolder(DATA_DIR + "/test", transform=test_transform)

# Nạp ảnh bằng nhiều tiến trình để GPU không phải chờ CPU
loader_options = {"num_workers": 8, "pin_memory": True, "persistent_workers": True}
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, **loader_options)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, **loader_options)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, **loader_options)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))




Train: 9542
Validation: 1685
Test: 864


In [5]:
# 4. Tạo CNN 3 blocks
class CNN(nn.Module):
    def __init__(self, number_of_classes):
        super().__init__()

        self.layers = nn.Sequential(
            # [B, 3, 224, 224] -> [B, 32, 112, 112]
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # [B, 32, 112, 112] -> [B, 64, 56, 56]
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # [B, 64, 56, 56] -> [B, 128, 28, 28]
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # [B, 128, 28, 28] -> [B, 128, 4, 4] -> [B, 2048]
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, number_of_classes)
        )

    def forward(self, images):
        return self.layers(images)


model = CNN(len(train_dataset.classes)).to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)




In [6]:
# 5. Train và Validation
best_val_accuracy = -1.0
best_weights = None
best_epoch = 0
patience = 15
epochs_without_improvement = 0

for epoch in range(50):
    model.train()
    train_loss = 0
    train_true_labels = []
    train_predicted_labels = []

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = loss_function(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * labels.size(0)
        predictions = outputs.argmax(dim=1)

        train_true_labels.extend(labels.cpu().tolist())
        train_predicted_labels.extend(predictions.cpu().tolist())

    train_loss /= len(train_dataset)
    train_metrics = calculate_metrics(train_true_labels, train_predicted_labels)
    train_accuracy = train_metrics["accuracy"]

    model.eval()
    val_loss = 0
    val_true_labels = []
    val_predicted_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = loss_function(outputs, labels)

            val_loss += loss.item() * labels.size(0)
            predictions = outputs.argmax(dim=1)

            val_true_labels.extend(labels.cpu().tolist())
            val_predicted_labels.extend(predictions.cpu().tolist())

    val_loss /= len(val_dataset)
    val_metrics = calculate_metrics(val_true_labels, val_predicted_labels)
    val_accuracy = val_metrics["accuracy"]

    # Sao chép trọng số sang RAM để các epoch sau không ghi đè lên bản tốt nhất.
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_weights = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
        best_epoch = epoch + 1
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    print(
        f"Epoch {epoch + 1}/50 | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_accuracy:.2%} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_accuracy:.2%}"
    )

    # Dừng khi validation accuracy không tăng trong 15 epoch liên tiếp.
    if epochs_without_improvement >= patience:
        print(f"Early stopping tại epoch {epoch + 1}")
        break

# Vòng lặp kết thúc khi model đang giữ trọng số của epoch cuối, nạp lại trọng số tốt nhất.
model.load_state_dict(best_weights)




Epoch 1/50 | Train Loss: 1.8326 | Train Acc: 34.09% | Val Loss: 1.6913 | Val Acc: 39.47%


Epoch 2/50 | Train Loss: 1.6546 | Train Acc: 40.83% | Val Loss: 1.6336 | Val Acc: 42.55%


Epoch 3/50 | Train Loss: 1.5876 | Train Acc: 44.25% | Val Loss: 1.5239 | Val Acc: 47.83%


Epoch 4/50 | Train Loss: 1.5280 | Train Acc: 46.37% | Val Loss: 1.4538 | Val Acc: 50.15%


Epoch 5/50 | Train Loss: 1.4672 | Train Acc: 48.52% | Val Loss: 1.4330 | Val Acc: 51.57%


...

Epoch 47/50 | Train Loss: 0.6938 | Train Acc: 75.94% | Val Loss: 1.1969 | Val Acc: 63.09%


Epoch 48/50 | Train Loss: 0.7217 | Train Acc: 74.75% | Val Loss: 1.0640 | Val Acc: 66.77%


Epoch 49/50 | Train Loss: 0.7129 | Train Acc: 74.87% | Val Loss: 1.0183 | Val Acc: 68.78%


Epoch 50/50 | Train Loss: 0.6846 | Train Acc: 75.60% | Val Loss: 1.0685 | Val Acc: 65.93%


<All keys matched successfully>

In [7]:
# 6. Test
print(f"Best epoch: {best_epoch} | Best Val Acc: {best_val_accuracy:.2%}")
model.eval()
true_labels = []
predicted_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        true_labels.extend(labels.cpu().tolist())
        predicted_labels.extend(predictions.cpu().tolist())

metrics = calculate_metrics(true_labels, predicted_labels)

print("\nTest Result")
print(f"Test Accuracy: {metrics['accuracy']:.2%}")
print(f"Macro Precision: {metrics['macro_precision']:.4f}")
print(f"Macro Recall: {metrics['macro_recall']:.4f}")
print(f"Macro F1: {metrics['macro_f1']:.4f}")
print(f"Weighted F1: {metrics['weighted_f1']:.4f}")
print()
print(classification_report(true_labels, predicted_labels, target_names=test_dataset.classes, digits=4, zero_division=0))

Best epoch: 45 | Best Val Acc: 69.14%



Test Result
Test Accuracy: 63.08%
Macro Precision: 0.6350
Macro Recall: 0.6272
Macro F1: 0.6253
Weighted F1: 0.6346

              precision    recall  f1-score   support

         Alu     0.8101    0.6465    0.7191        99
      Carton     0.8462    0.7130    0.7739       108
    Foam_box     0.5278    0.4368    0.4780        87
    Milk_box     0.7677    0.7525    0.7600       101
       Other     0.4429    0.6200    0.5167        50
         PET     0.4931    0.6961    0.5772       102
       Paper     0.6923    0.6792    0.6857       106
   Paper_cup     0.6733    0.6476    0.6602       105
 Plastic_cup     0.4615    0.4528    0.4571       106

    accuracy                         0.6308       864
   macro avg     0.6350    0.6272    0.6253       864
weighted avg     0.6487    0.6308    0.6346       864



In [8]:
# 7. Lưu model
torch.save(model.state_dict(), CHECKPOINT_DIR + "/m1_03.pt")